# Домашняя работа: анализ данных службы доставки с NumPy

**Дисциплина:** Анализ данных и основы машинного обучения  
**Тема:** NumPy и анализ числового датасета

## Цель работы

Самостоятельно провести анализ индивидуального набора данных средствами NumPy: исследовать структуру, проверить качество, очистить ошибки, использовать логические маски, статистические показатели, сравнение групп, процентили, корреляции и поиск проблемных наблюдений.


**Вариант: 5.** Используется уже сгенерированный файл `delivery_data.npy`
размером 120 × 6. Его точное совпадение с результатом
`generate_dataset(5)` проверено с учётом `NaN`.

Во всех результатах **индекс строки начинается с 0**. Если указан номер
доставки, он равен индексу + 1. Задания 4–10 выполняются по очищенному
массиву `prepared_data`; исходный `data` сохраняется без изменений.


## Получение данных

1. Запустите `generate_delivery_data.py`.
2. Введите свой номер варианта.
3. После запуска появятся `delivery_data.npy` и `delivery_data.csv`.
4. Для выполнения работы используйте `delivery_data.npy`.
5. Поместите `delivery_data.npy` и этот ноутбук в одну папку.

> При проверке преподаватель может повторно создать ваш датасет по номеру варианта.


## Структура датасета

| Индекс | Показатель | Единица |
|---:|---|---|
| 0 | Расстояние доставки | км |
| 1 | Время доставки | мин |
| 2 | Стоимость заказа | руб. |
| 3 | Время ожидания курьера | мин |
| 4 | Оценка клиента | от 1 до 5 |
| 5 | Тип доставки | 0 — обычная, 1 — экспресс |

В данных могут встречаться пропуски, бесконечные значения, физически некорректные значения и статистически необычные наблюдения.


## Подготовка

In [1]:
import numpy as np

variant = 5
data = np.load("delivery_data.npy")
columns = ["distance", "duration", "order_amount",
           "wait_time", "rating", "delivery_type"]

np.set_printoptions(precision=4, suppress=True, linewidth=120)
print("Вариант:", variant)
print("Первые 5 доставок:")
print(data[:5])


Вариант: 5
Первые 5 доставок:
[[  28.94   84.47 2650.7    13.28    4.28    1.  ]
 [   7.17   37.34 3510.9     5.11    4.16    1.  ]
 [  13.85     nan 2296.88    4.51    4.57    1.  ]
 [   8.59   30.11 1303.85    8.25    4.11    1.  ]
 [  22.01   54.03 1293.31    0.      4.99    1.  ]]


---
# Задание 1. Первичное исследование данных

Определите тип объекта, размерность, `shape`, количество элементов, `dtype`, объём памяти массива, количество доставок и количество признаков.

Получите каждый столбец в отдельный одномерный массив: `distance`, `duration`, `order_amount`, `wait_time`, `rating`, `delivery_type`. Выведите первые 10 значений каждого.

**Письменный вопрос:** какие признаки являются непрерывными, а какой можно считать категориальным, несмотря на числовое представление?

In [2]:
print("Тип объекта:", type(data))
print("Размерность:", data.ndim)
print("Shape:", data.shape)
print("Количество элементов:", data.size)
print("Тип данных (dtype):", data.dtype)
print("Объём памяти (байт):", data.nbytes)

print("Количество доставок:", data.shape[0])
print("Количество признаков:", data.shape[1])

Тип объекта: <class 'numpy.ndarray'>
Размерность: 2
Shape: (120, 6)
Количество элементов: 720
Тип данных (dtype): float64
Объём памяти (байт): 5760
Количество доставок: 120
Количество признаков: 6


In [3]:
distance = data[:, 0]
duration = data[:, 1]
order_amount = data[:, 2]
wait_time = data[:, 3]
rating = data[:, 4]
delivery_type = data[:, 5]

print("distance:", distance[:10])
print("duration:", duration[:10])
print("order_amount:", order_amount[:10])
print("wait_time:", wait_time[:10])
print("rating:", rating[:10])
print("delivery_type:", delivery_type[:10])

distance: [28.94  7.17 13.85  8.59 22.01 17.69 22.88  8.09 12.64 20.05]
duration: [84.47 37.34   nan 30.11 54.03 48.   61.17 34.61 39.61 47.18]
order_amount: [2650.7  3510.9  2296.88 1303.85 1293.31 3204.48 6279.27 2610.   6146.8  2510.14]
wait_time: [13.28  5.11  4.51  8.25  0.    7.55  5.02  6.39  6.42  6.28]
rating: [4.28 4.16 4.57 4.11 4.99 8.   4.14 4.28 4.21 4.68]
delivery_type: [1. 1. 1. 1. 1. 0. 0. 1. 1. 1.]


**Ответ.** `distance`, `duration`, `order_amount` и `wait_time` —
количественные признаки, которые при анализе можно считать непрерывными
(в файле они округлены до сотых). `rating` — ограниченная числовая оценка
от 1 до 5; в этом датасете она дробная, например 4,28, поэтому её также
анализируем как непрерывный количественный показатель. Это не шкала только
из пяти целых значений.

`delivery_type` — бинарный категориальный признак: 0 обозначает обычную,
1 — экспресс-доставку. Его тип `float64` обусловлен хранением в общем
массиве и не делает категории непрерывными величинами.


---
# Задание 2. Проверка качества данных

Проведите полный аудит исходного массива.

Определите количество `NaN`, `inf`, `-inf` и координаты проблемных элементов.

Проверьте ограничения:

- `distance > 0`
- `duration > 0`
- `order_amount > 0`
- `wait_time >= 0`
- `1 <= rating <= 5`
- `delivery_type` равен только `0` или `1`

Напишите функцию `check_delivery_data(data)`, которая для каждой проблемы выводит номер доставки, название признака и исходное значение.

In [4]:
def delivery_constraints(array):
    """Маска выполнения ограничений для каждого элемента массива."""
    return np.column_stack((
        array[:, 0] > 0,
        array[:, 1] > 0,
        array[:, 2] > 0,
        array[:, 3] >= 0,
        (array[:, 4] >= 1) & (array[:, 4] <= 5),
        np.isin(array[:, 5], [0, 1]),
    ))


def check_delivery_data(data):
    """Печатает аудит, возвращает маски ошибок и не изменяет data."""
    if data.ndim != 2 or data.shape[1] != 6:
        raise ValueError("Ожидается двумерный массив с 6 столбцами")

    nan_mask = np.isnan(data)
    posinf_mask = np.isposinf(data)
    neginf_mask = np.isneginf(data)
    # Неконечные значения считаем отдельно от конечных нарушений границ.
    invalid_mask = np.isfinite(data) & ~delivery_constraints(data)
    problem_mask = ~np.isfinite(data) | invalid_mask

    for name, mask in (("NaN", nan_mask), ("inf", posinf_mask),
                       ("-inf", neginf_mask),
                       ("Конечные значения вне ограничений", invalid_mask)):
        print(f"{name}: {np.count_nonzero(mask)}")
        print("Координаты (индекс строки, индекс столбца):")
        print(np.argwhere(mask))

    print("Подробный список проблем:")
    for row, col in np.argwhere(problem_mask):
        value = data[row, col]
        if np.isnan(value):
            reason = "пропуск"
        elif np.isinf(value):
            reason = "бесконечность"
        else:
            reason = "нарушение ограничения"
        print(f"Доставка № {row + 1} (индекс {row}), "
              f"признак {columns[col]}, значение {value}: {reason}")

    print("Всего проблемных элементов:", np.count_nonzero(problem_mask))
    print("Строк с проблемами:", np.count_nonzero(problem_mask.any(axis=1)))
    return {
        "nan_mask": nan_mask,
        "posinf_mask": posinf_mask,
        "neginf_mask": neginf_mask,
        "invalid_mask": invalid_mask,
        "problem_mask": problem_mask,
    }


audit = check_delivery_data(data)


NaN: 4
Координаты (индекс строки, индекс столбца):
[[ 2  1]
 [82  0]
 [85  4]
 [96  4]]
inf: 1
Координаты (индекс строки, индекс столбца):
[[101   2]]
-inf: 0
Координаты (индекс строки, индекс столбца):
[]
Конечные значения вне ограничений: 5
Координаты (индекс строки, индекс столбца):
[[  5   4]
 [ 25   2]
 [ 73   0]
 [108   1]
 [115   3]]
Подробный список проблем:
Доставка № 3 (индекс 2), признак duration, значение nan: пропуск
Доставка № 6 (индекс 5), признак rating, значение 8.0: нарушение ограничения
Доставка № 26 (индекс 25), признак order_amount, значение -3500.0: нарушение ограничения
Доставка № 74 (индекс 73), признак distance, значение -12.0: нарушение ограничения
Доставка № 83 (индекс 82), признак distance, значение nan: пропуск
Доставка № 86 (индекс 85), признак rating, значение nan: пропуск
Доставка № 97 (индекс 96), признак rating, значение nan: пропуск
Доставка № 102 (индекс 101), признак order_amount, значение inf: бесконечность
Доставка № 109 (индекс 108), признак dura

Найдено **4 пропуска `NaN`, 1 значение `inf`, 0 значений `-inf`**
и **5 конечных значений вне допустимых границ**. Всего 10 проблемных
элементов в 10 разных строках. Это ошибки качества данных;
положительные, но необычно большие значения будут исследоваться отдельно.


---
# Задание 3. Подготовка данных к анализу

Создайте копию исходного массива. Исходный `data` изменять нельзя.

В копии замените `inf` и физически некорректные значения на `NaN`. Для числовых признаков заполните пропуски средним значением столбца. Для `delivery_type` среднее использовать нельзя: если встретится пропуск, замените его наиболее частым значением `0` или `1`.

Сохраните результат в `prepared_data` и программно докажите, что `NaN` и `inf` отсутствуют, а все значения удовлетворяют ограничениям.

In [5]:
def prepare_delivery_data(data):
    """Очищает независимую копию; числовые пропуски — средним, тип — модой."""
    prepared = np.array(data, dtype=float, copy=True)
    if prepared.ndim != 2 or prepared.shape[1] != 6 or len(prepared) == 0:
        raise ValueError("Ожидается непустой двумерный массив с 6 столбцами")

    invalid = ~np.isfinite(prepared) | ~delivery_constraints(prepared)
    prepared[invalid] = np.nan

    # Сначала убраны ВСЕ ошибки; средние считаем только по валидным данным.
    for col in range(5):
        missing = np.isnan(prepared[:, col])
        if missing.all():
            raise ValueError(f"Нет допустимых значений в столбце {columns[col]}")
        prepared[missing, col] = np.nanmean(prepared[:, col])

    types = prepared[:, 5]
    valid_types = types[~np.isnan(types)].astype(int)
    if valid_types.size == 0:
        raise ValueError("Невозможно определить моду: нет допустимых типов доставки")
    counts = np.bincount(valid_types, minlength=2)
    mode = np.argmax(counts)  # При равенстве частот выбирается 0.
    types[np.isnan(types)] = mode

    assert np.isfinite(prepared).all()
    assert delivery_constraints(prepared).all()
    return prepared


original_data = data.copy()
prepared_data = prepare_delivery_data(data)

print("Количество NaN:", np.count_nonzero(np.isnan(prepared_data)))
print("Количество inf и -inf:", np.count_nonzero(np.isinf(prepared_data)))
print("Исправлено элементов:", np.count_nonzero(audit["problem_mask"]))
print("Средние для заполнения первых пяти столбцов:")
print(prepared_data[:, :5].mean(axis=0))
print("Частоты типов доставки:",
      np.bincount(prepared_data[:, 5].astype(int), minlength=2))

assert np.array_equal(data, original_data, equal_nan=True)
assert not np.shares_memory(data, prepared_data)
assert np.isfinite(prepared_data).all()
assert delivery_constraints(prepared_data).all()

print("Все ограничения выполнены; исходный массив не изменён.")
print("Размер подготовленного массива:", prepared_data.shape)


Количество NaN: 0
Количество inf и -inf: 0
Исправлено элементов: 10
Средние для заполнения первых пяти столбцов:
[  15.517    48.153  3597.7334    7.9352    4.495 ]
Частоты типов доставки: [52 68]
Все ограничения выполнены; исходный массив не изменён.
Размер подготовленного массива: (120, 6)


После подготовки остаются все 120 доставок. Пропуски и некорректные
значения заменены средними допустимых значений соответствующих столбцов.
Пропусков типа доставки в варианте 5 нет; функция всё равно предусматривает
их заполнение модой. Если целый столбец не содержит допустимых значений,
функция явно сообщает об ошибке: среднее или моду в таком случае определить нельзя.

Доставка за **180 минут** и заказ на **35 000 рублей** оставлены в анализе:
верхние границы времени и суммы в задании не установлены. Статистическая
необычность сама по себе не является физической некорректностью.
Заполнение средними сохраняет объём данных, но может искажать связи между
признаками и производные показатели отдельных строк.


---
# Задание 4. Обычная и экспресс-доставка

Разделите данные на обычные (`delivery_type == 0`) и экспресс-доставки (`delivery_type == 1`).

Для каждой группы вычислите количество доставок, среднее расстояние, среднее и медианное время доставки, среднее ожидание, среднюю оценку и среднюю стоимость заказа.

**Ответьте письменно:** какой тип быстрее, на сколько минут отличаются средние времена, отличается ли средняя оценка и можно ли считать тип доставки единственной причиной различий?

In [6]:
def summarize_deliveries(rows):
    """Статистики группы; для пустой группы средние и медиана не определены."""
    count = len(rows)
    means = rows[:, :5].mean(axis=0) if count else np.full(5, np.nan)
    return {
        "count": count,
        "mean_distance": means[0],
        "mean_duration": means[1],
        "median_duration": np.median(rows[:, 1]) if count else np.nan,
        "mean_wait_time": means[3],
        "mean_rating": means[4],
        "mean_order_amount": means[2],
    }


def print_group_statistics(title, rows):
    stats = summarize_deliveries(rows)
    print(f"\n{title}")
    print("Количество доставок:", stats["count"])
    print(f"Среднее расстояние: {stats['mean_distance']:.2f} км")
    print(f"Среднее время: {stats['mean_duration']:.2f} мин")
    print(f"Медианное время: {stats['median_duration']:.2f} мин")
    print(f"Среднее ожидание: {stats['mean_wait_time']:.2f} мин")
    print(f"Средняя оценка: {stats['mean_rating']:.3f}")
    print(f"Средняя стоимость заказа: {stats['mean_order_amount']:.2f} руб.")


regular = prepared_data[prepared_data[:, 5] == 0]
express = prepared_data[prepared_data[:, 5] == 1]
print_group_statistics("Обычная доставка", regular)
print_group_statistics("Экспресс-доставка", express)

regular_stats = summarize_deliveries(regular)
express_stats = summarize_deliveries(express)
print("\nРазница средних времён (обычная − экспресс): "
      f"{regular_stats['mean_duration'] - express_stats['mean_duration']:.2f} мин")
print("Разница средних оценок (экспресс − обычная): "
      f"{express_stats['mean_rating'] - regular_stats['mean_rating']:.3f}")



Обычная доставка
Количество доставок: 52
Среднее расстояние: 18.00 км
Среднее время: 57.29 мин
Медианное время: 60.61 мин
Среднее ожидание: 8.37 мин
Средняя оценка: 4.454
Средняя стоимость заказа: 3693.28 руб.

Экспресс-доставка
Количество доставок: 68
Среднее расстояние: 13.62 км
Среднее время: 41.17 мин
Медианное время: 36.91 мин
Среднее ожидание: 7.60 мин
Средняя оценка: 4.526
Средняя стоимость заказа: 3524.67 руб.

Разница средних времён (обычная − экспресс): 16.12 мин
Разница средних оценок (экспресс − обычная): 0.072


**Ответ.** Обычных доставок — **52**, экспресс — **68**. Экспресс в среднем
быстрее: **41,17 против 57,29 минуты**, разница **16,12 минуты**. Медианы
равны 36,91 и 60,61 минуты соответственно. Средняя оценка экспресс-доставки
немного выше: **4,526 против 4,454**, разница около **0,072 балла**.

Тип доставки нельзя считать единственной причиной различий: среднее
расстояние экспресс-доставки тоже меньше (**13,62 против 18,00 км**),
а ожидание составляет 7,60 против 8,37 минуты. На среднее обычных доставок
также влияет наблюдение с временем 180 минут. Для причинного вывода нужны
сопоставимые по расстоянию и другим условиям группы либо эксперимент.


---
# Задание 5. Скорость доставки

Создайте показатель:

`speed = distance / (duration / 60)`

Вычисление выполните без цикла `for`.

Определите среднюю, медианную, минимальную и максимальную скорость; индексы самой медленной и самой быстрой доставки; выведите полные строки этих наблюдений.

Найдите доставки, где `speed > 60 км/ч`.

**Письменный вопрос:** насколько реалистичны такие скорости для городской доставки?

In [7]:
# Векторное вычисление: ни деление, ни поиск скоростей не требуют цикла for.
speed = prepared_data[:, 0] / (prepared_data[:, 1] / 60)
slowest_index = int(np.argmin(speed))
fastest_index = int(np.argmax(speed))
fast_mask = speed > 60

print(f"Средняя скорость: {speed.mean():.2f} км/ч")
print(f"Медианная скорость: {np.median(speed):.2f} км/ч")
print(f"Минимальная скорость: {speed.min():.2f} км/ч")
print(f"Максимальная скорость: {speed.max():.2f} км/ч")
print("Индекс самой медленной доставки:", slowest_index)
print("Полная строка:", prepared_data[slowest_index])
print("Индекс самой быстрой доставки:", fastest_index)
print("Полная строка:", prepared_data[fastest_index])
print("Количество доставок быстрее 60 км/ч:", np.count_nonzero(fast_mask))
print("Индексы:", np.flatnonzero(fast_mask))
print("Строки:\n", prepared_data[fast_mask])
print("Скорости:", speed[fast_mask])


Средняя скорость: 18.51 км/ч
Медианная скорость: 19.50 км/ч
Минимальная скорость: 2.55 км/ч
Максимальная скорость: 56.67 км/ч
Индекс самой медленной доставки: 108
Полная строка: [   2.05    48.153 1831.12    12.43     4.32     1.   ]
Индекс самой быстрой доставки: 73
Полная строка: [  15.517   16.43  4761.51     6.9      4.       1.   ]
Количество доставок быстрее 60 км/ч: 0
Индексы: []
Строки:
 []
Скорости: []


**Ответ.** Средняя скорость равна **18,51 км/ч**, медианная — **19,50 км/ч**.
Минимальная — **2,55 км/ч** (индекс **108**), максимальная — **56,67 км/ч**
(индекс **73**). Доставок со скоростью выше 60 км/ч нет.

Это средняя скорость выполнения заказа, рассчитанная по полному времени
доставки, а не мгновенная скорость движения курьера. Среднее значение
свыше 60 км/ч для городской доставки с остановками было бы необычным и
требовало бы проверки расстояния, времени и условий маршрута.

Оба экстремума требуют осторожности: в строке 108 исходное время −25 минут
заменено средним, а в строке 73 средним заменено расстояние −12 км.
Поэтому по этим строкам нельзя достоверно назвать реальных самого
медленного и самого быстрого курьеров.


---
# Задание 6. Анализ задержек

Вычислите:

`expected_time = 15 + distance * 2`

`delay = duration - expected_time`

Отрицательную задержку считать равной нулю.

Определите количество доставок без задержки и с задержкой, среднюю и максимальную задержку, индекс максимальной задержки.

Найдите доставки с `delay > 20`, вычислите для них среднюю оценку и сравните со средней оценкой по всему датасету.

**Письменный вопрос:** есть ли признаки связи задержек с удовлетворённостью клиентов?

In [8]:
expected_time = 15 + prepared_data[:, 0] * 2
delay = np.maximum(prepared_data[:, 1] - expected_time, 0)
delayed_mask = delay > 0
long_delay_mask = delay > 20
max_delay_index = int(np.argmax(delay))

print("Доставок без задержки:", np.count_nonzero(delay == 0))
print("Доставок с задержкой:", np.count_nonzero(delayed_mask))
print(f"Средняя задержка по всем доставкам: {delay.mean():.2f} мин")
if delayed_mask.any():
    print(f"Средняя задержка только среди опоздавших: {delay[delayed_mask].mean():.2f} мин")
print(f"Максимальная задержка: {delay.max():.2f} мин")
print("Индекс максимальной задержки:", max_delay_index)
print("Количество задержек более 20 минут:", np.count_nonzero(long_delay_mask))
print("Индексы:", np.flatnonzero(long_delay_mask))
print("Полные строки:\n", prepared_data[long_delay_mask])
print("Задержки:", delay[long_delay_mask])
print(f"Средняя оценка по всему датасету: {prepared_data[:, 4].mean():.3f}")
if long_delay_mask.any():
    print(f"Средняя оценка при задержке > 20: {prepared_data[long_delay_mask, 4].mean():.3f}")
else:
    print("Задержек более 20 минут нет; средняя оценка группы не определена.")


Доставок без задержки: 58
Доставок с задержкой: 62
Средняя задержка по всем доставкам: 4.40 мин
Средняя задержка только среди опоздавших: 8.51 мин
Максимальная задержка: 106.96 мин
Индекс максимальной задержки: 105
Количество задержек более 20 минут: 2
Индексы: [105 108]
Полные строки:
 [[  29.02   180.    6910.11     2.46     4.76     0.   ]
 [   2.05    48.153 1831.12    12.43     4.32     1.   ]]
Задержки: [106.96   29.053]
Средняя оценка по всему датасету: 4.495
Средняя оценка при задержке > 20: 4.540


**Ответ.** Без задержки выполнены **58** доставок, с задержкой — **62**.
Средняя задержка по всем заказам равна **4,40 минуты**; среди опоздавших —
**8,51 минуты**. Максимальная задержка — **106,96 минуты**, индекс **105**.

Более чем на 20 минут опоздали две доставки: индексы **105** и **108**.
Их средняя оценка — **4,540**, по всему датасету — **4,495**. По такому
сравнению ухудшение оценки при больших задержках не наблюдается.
Однако два наблюдения — слишком малая группа для устойчивого вывода,
а задержка строки 108 рассчитана по заполненному средним времени.
Нельзя заключить, что задержки не влияют на удовлетворённость клиентов.
Кроме того, ожидаемое время здесь задаёт учебная формула, а не обещанный
клиенту срок доставки.


---
# Задание 7. Анализ стоимости заказов с использованием процентилей

Рассчитайте 25-й, 50-й, 75-й и 90-й процентили стоимости заказа.

Разделите заказы на четыре группы: до 25-го процентиля, 25–50, 50–75 и выше 75-го.

Для каждой группы вычислите количество заказов, среднее время доставки, среднее расстояние и среднюю оценку.

Дополнительно найдите 10% самых дорогих заказов и вычислите для них среднюю стоимость, расстояние и время доставки.

**Письменный вопрос:** видно ли по вашему варианту, что дорогие заказы доставляются принципиально иначе?

In [9]:
amounts = prepared_data[:, 2]
p25, p50, p75, p90 = np.percentile(amounts, [25, 50, 75, 90])
print(f"25-й процентиль: {p25:.2f} руб.")
print(f"50-й процентиль: {p50:.2f} руб.")
print(f"75-й процентиль: {p75:.2f} руб.")
print(f"90-й процентиль: {p90:.2f} руб.")

# Непересекающиеся интервалы: значение на границе относим к нижней группе.
quartile_masks = [
    amounts <= p25,
    (amounts > p25) & (amounts <= p50),
    (amounts > p50) & (amounts <= p75),
    amounts > p75,
]
group_names = ["До 25-го включительно", "Выше 25-го до 50-го включительно",
               "Выше 50-го до 75-го включительно", "Выше 75-го"]
assert (np.column_stack(quartile_masks).sum(axis=1) == 1).all()

for name, mask in zip(group_names, quartile_masks):
    rows = prepared_data[mask]
    print(f"\n{name}: {len(rows)} заказов")
    if len(rows):
        print(f"Среднее время: {rows[:, 1].mean():.2f} мин; "
              f"расстояние: {rows[:, 0].mean():.2f} км; "
              f"оценка: {rows[:, 4].mean():.3f}")

# В данном варианте на границе p90 нет повторов: выбраны ровно 12 из 120.
# В других данных повторы на границе могут увеличить долю выбранных заказов.
top10_mask = amounts >= p90
top10 = prepared_data[top10_mask]
print("\n10% самых дорогих заказов:", len(top10))
print(f"Фактическая доля: {len(top10) / len(prepared_data):.1%}")
print(f"Средняя стоимость: {top10[:, 2].mean():.2f} руб.")
print(f"Среднее расстояние: {top10[:, 0].mean():.2f} км")
print(f"Среднее время: {top10[:, 1].mean():.2f} мин")


25-й процентиль: 2225.51 руб.
50-й процентиль: 3090.18 руб.
75-й процентиль: 4425.89 руб.
90-й процентиль: 5435.85 руб.

До 25-го включительно: 30 заказов
Среднее время: 43.65 мин; расстояние: 13.79 км; оценка: 4.547

Выше 25-го до 50-го включительно: 30 заказов
Среднее время: 48.70 мин; расстояние: 15.93 км; оценка: 4.508

Выше 50-го до 75-го включительно: 30 заказов
Среднее время: 51.47 мин; расстояние: 17.10 км; оценка: 4.472

Выше 75-го: 30 заказов
Среднее время: 48.79 мин; расстояние: 15.25 км; оценка: 4.453

10% самых дорогих заказов: 12
Фактическая доля: 10.0%
Средняя стоимость: 8865.85 руб.
Среднее расстояние: 15.51 км
Среднее время: 53.79 мин


**Ответ.** Процентили стоимости: **2 225,51; 3 090,18; 4 425,89;
5 435,85 рубля**. В каждом квартиле по **30** заказов. Среднее время
по квартилям равно 43,65; 48,70; 51,47; 48,79 минуты. Устойчивого
монотонного ускорения или замедления с ростом стоимости нет.

В верхние 10% попали **12** заказов: средняя стоимость **8 865,85 рубля**,
расстояние **15,51 км**, время **53,79 минуты**. Для всего датасета
расстояние равно 15,52 км, время — 48,15 минуты. Верхняя группа несколько
медленнее, но в ней находится доставка за 180 минут; кроме того, заказ
на 35 000 рублей повышает среднюю стоимость. Эти различия сами по себе
не доказывают принципиально иной способ доставки дорогих заказов.
Корреляция стоимости и времени в задании 9 также близка к нулю.


---
# Задание 8. Поиск проблемных доставок

Доставка считается потенциально проблемной, если выполняются хотя бы два условия:

- `duration > 90`
- `wait_time > 15`
- `rating < 3.5`
- `delay > 20`

Для каждой строки посчитайте количество выполненных условий и найдите строки, где `problem_count >= 2`.

Для проблемных доставок вычислите количество, среднее расстояние, среднее время, среднее ожидание, среднюю оценку и среднюю стоимость. Сравните с общим датасетом.

In [10]:
problem_conditions = np.column_stack((
    prepared_data[:, 1] > 90,
    prepared_data[:, 3] > 15,
    prepared_data[:, 4] < 3.5,
    delay > 20,
))
problem_count = np.sum(problem_conditions, axis=1)
problem_mask = problem_count >= 2
problematic = prepared_data[problem_mask]

print("Количество выполненных условий для каждой строки:")
print(problem_count)
print("Число срабатываний по условиям (время, ожидание, оценка, задержка):")
print(problem_conditions.sum(axis=0))
print("Индексы проблемных доставок:", np.flatnonzero(problem_mask))
print("Полные строки:\n", problematic)
print_group_statistics("Проблемные доставки", problematic)
print_group_statistics("Весь датасет", prepared_data)


Количество выполненных условий для каждой строки:
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 2 0 0 1 0 0 0 0 0 0 0 0 0
 0 0]
Число срабатываний по условиям (время, ожидание, оценка, задержка):
[2 1 0 2]
Индексы проблемных доставок: [105]
Полные строки:
 [[  29.02  180.   6910.11    2.46    4.76    0.  ]]

Проблемные доставки
Количество доставок: 1
Среднее расстояние: 29.02 км
Среднее время: 180.00 мин
Медианное время: 180.00 мин
Среднее ожидание: 2.46 мин
Средняя оценка: 4.760
Средняя стоимость заказа: 6910.11 руб.

Весь датасет
Количество доставок: 120
Среднее расстояние: 15.52 км
Среднее время: 48.15 мин
Медианное время: 44.66 мин
Среднее ожидание: 7.94 мин
Средняя оценка: 4.495
Средняя стоимость заказа: 3597.73 руб.


Выявлена **1 потенциально проблемная доставка**, индекс **105** (номер 106).
Выполнены два условия: время **180 минут > 90** и задержка
**106,96 минуты > 20**. Её расстояние **29,02 км** против средних 15,52 км,
время — **180,00 против 48,15 минуты**, ожидание — **2,46 против 7,94 минуты**,
оценка — **4,760 против 4,495**, стоимость — **6 910,11 против 3 597,73 рубля**.

Долгая доставка не сопровождается низкой оценкой или большим ожиданием.
Условия времени и задержки связаны между собой: их одновременное
выполнение не является двумя независимыми доказательствами проблемы.
Доставка 108 имеет задержку более 20 минут, но удовлетворяет только одному
условию и потому в проблемную группу не попадает.


---
# Задание 9. Исследование взаимосвязей

Для признаков `distance`, `duration`, `order_amount`, `wait_time`, `rating` рассчитайте корреляционную матрицу.

Определите:

1. пару с максимальной положительной корреляцией;
2. пару с наиболее сильной отрицательной корреляцией;
3. корреляцию расстояния и времени доставки;
4. корреляцию ожидания и оценки;
5. корреляцию времени доставки и оценки.

Дополнительно вычислите ковариационную матрицу.

**Обязательный письменный вопрос:** почему высокая корреляция не доказывает причинно-следственную связь?

In [11]:
def print_labeled_matrix(matrix, labels, precision=4):
    print(" " * 15 + " ".join(f"{name:>14}" for name in labels))
    for name, row in zip(labels, matrix):
        print(f"{name:>14} " + " ".join(f"{value:14.{precision}f}" for value in row))


numeric_columns = columns[:5]
correlation_matrix = np.corrcoef(prepared_data[:, :5], rowvar=False)
covariance_matrix = np.cov(prepared_data[:, :5], rowvar=False, ddof=1)

print("Корреляционная матрица Пирсона:")
print_labeled_matrix(correlation_matrix, numeric_columns)
print("\nВыборочная ковариационная матрица (делитель n − 1):")
print_labeled_matrix(covariance_matrix, numeric_columns)

# Только уникальные пары разных признаков: исключаем диагональ и повторы.
upper_i, upper_j = np.triu_indices(len(numeric_columns), k=1)
pair_correlations = correlation_matrix[upper_i, upper_j]
positive_index = int(np.argmax(pair_correlations))
negative_index = int(np.argmin(pair_correlations))
positive_pair = (upper_i[positive_index], upper_j[positive_index])
negative_pair = (upper_i[negative_index], upper_j[negative_index])

print("\nМаксимальная положительная корреляция:",
      numeric_columns[positive_pair[0]], "и", numeric_columns[positive_pair[1]],
      f"= {correlation_matrix[positive_pair]:.4f}")
print("Наиболее сильная отрицательная корреляция:",
      numeric_columns[negative_pair[0]], "и", numeric_columns[negative_pair[1]],
      f"= {correlation_matrix[negative_pair]:.4f}")
print(f"Расстояние и время доставки: {correlation_matrix[0, 1]:.4f}")
print(f"Ожидание и оценка: {correlation_matrix[3, 4]:.4f}")
print(f"Время доставки и оценка: {correlation_matrix[1, 4]:.4f}")


Корреляционная матрица Пирсона:
                     distance       duration   order_amount      wait_time         rating
      distance         1.0000         0.8853         0.0076         0.0487        -0.1741
      duration         0.8853         1.0000         0.0175         0.0933        -0.1928
  order_amount         0.0076         0.0175         1.0000        -0.0980         0.0359
     wait_time         0.0487         0.0933        -0.0980         1.0000        -0.3400
        rating        -0.1741        -0.1928         0.0359        -0.3400         1.0000

Выборочная ковариационная матрица (делитель n − 1):
                     distance       duration   order_amount      wait_time         rating
      distance        80.9097       198.6348       224.9813         1.6130        -0.4594
      duration       198.6348       622.2636      1427.0198         8.5731        -1.4103
  order_amount       224.9813      1427.0198  10726971.0834     -1183.1744        34.4756
     wait_time 

**Ответ.** Самая сильная положительная корреляция — между расстоянием
и временем доставки: **r = 0,8853**. Самая сильная отрицательная — между
ожиданием и оценкой: **r = −0,3400**. Между временем доставки и оценкой
наблюдается слабая отрицательная линейная связь: **r = −0,1928**.
Между стоимостью заказа и временем связи почти нет: **r = 0,0175**.

Высокая корреляция не доказывает причинно-следственную связь. Два признака
могут зависеть от третьего фактора (например, пробок, сложности маршрута
или времени суток); возможны обратная причинность и случайное совпадение.
Корреляция симметрична и не устанавливает направление влияния. Для
причинных выводов нужны дополнительные предпосылки, учёт смешивающих
факторов или контролируемый эксперимент.

Корреляция Пирсона описывает линейную связь и чувствительна к выбросам;
заполнение пропусков средними также влияет на оценки связи. Ковариация
зависит от единиц измерения, поэтому её величины для разных пар нельзя
напрямую сравнивать по силе связи. Для ковариационной матрицы использован
выборочный делитель **n − 1**.


---
# Задание 10. Итоговый аналитический отчёт

Напишите функцию `analyze_delivery_dataset(data)`.

Исходный массив функция изменять не должна.

Функция должна:

1. определить размер датасета;
2. найти `NaN`, `inf`, `-inf`;
3. найти физически некорректные данные;
4. очистить копию массива;
5. заполнить пропуски;
6. рассчитать основные статистики;
7. сравнить обычную и экспресс-доставку;
8. рассчитать скорости;
9. определить задержки;
10. найти проблемные доставки;
11. вывести корреляционную матрицу.

После отчёта напишите **не менее пяти содержательных выводов** по своему варианту.

In [12]:
def analyze_delivery_dataset(data):
    """Полный отчёт по данным доставки; исходный массив не изменяется.

    Используются функции аудита, подготовки и вывода из заданий 2–9.
    Возвращает словарь результатов для последующего анализа.
    """
    source = np.asarray(data)
    if source.ndim != 2 or source.shape[1] != 6 or len(source) == 0:
        raise ValueError("Ожидается непустой двумерный массив с 6 столбцами")
    source_before = source.copy()

    print("АНАЛИТИЧЕСКИЙ ОТЧЁТ")
    print("Размер датасета:", source.shape)
    print("Количество доставок:", source.shape[0])
    print("Количество признаков:", source.shape[1])
    print("Количество элементов:", source.size)
    print("dtype:", source.dtype, "; объём памяти:", source.nbytes, "байт")

    print("\nАУДИТ ИСХОДНЫХ ДАННЫХ")
    quality = check_delivery_data(source)
    cleaned = prepare_delivery_data(source)
    print("\nПосле очистки NaN:", np.isnan(cleaned).sum(),
          "; бесконечностей:", np.isinf(cleaned).sum())
    print("Все ограничения выполнены:", delivery_constraints(cleaned).all())

    print("\nОСНОВНЫЕ СТАТИСТИКИ")
    # Категориальный delivery_type описываем частотами, не средними.
    numeric = cleaned[:, :5]
    statistics = np.column_stack((
        numeric.mean(axis=0), np.median(numeric, axis=0),
        numeric.min(axis=0), numeric.max(axis=0), numeric.std(axis=0),
    ))
    print("Столбцы: среднее, медиана, минимум, максимум, std (ddof=0)")
    for name, row in zip(columns[:5], statistics):
        print(f"{name:>14}:", row)
    type_counts = np.bincount(cleaned[:, 5].astype(int), minlength=2)
    print("Частоты delivery_type (0, 1):", type_counts)

    print("\nСРАВНЕНИЕ ТИПОВ ДОСТАВКИ")
    groups = {}
    for delivery_kind, title in ((0, "Обычная"), (1, "Экспресс")):
        rows = cleaned[cleaned[:, 5] == delivery_kind]
        groups[delivery_kind] = summarize_deliveries(rows)
        print_group_statistics(title, rows)
    if groups[0]["count"] and groups[1]["count"]:
        difference = groups[0]["mean_duration"] - groups[1]["mean_duration"]
        print(f"Разница времени (обычная − экспресс): {difference:.2f} мин")
        rating_difference = groups[1]["mean_rating"] - groups[0]["mean_rating"]
        print(f"Разница оценок (экспресс − обычная): {rating_difference:.3f}")

    speeds = cleaned[:, 0] / (cleaned[:, 1] / 60)
    slowest = int(np.argmin(speeds))
    fastest = int(np.argmax(speeds))
    print("\nСКОРОСТИ (км/ч)")
    print(f"Средняя: {speeds.mean():.2f}; медианная: {np.median(speeds):.2f}; "
          f"минимум: {speeds.min():.2f}; максимум: {speeds.max():.2f}")
    print("Индекс самой медленной:", slowest, "; строка:", cleaned[slowest])
    print("Индекс самой быстрой:", fastest, "; строка:", cleaned[fastest])
    print("Индексы доставок со скоростью > 60:", np.flatnonzero(speeds > 60))

    expected = 15 + 2 * cleaned[:, 0]
    delays = np.maximum(cleaned[:, 1] - expected, 0)
    long_delays = delays > 20
    print("\nЗАДЕРЖКИ")
    print("Без задержки:", np.count_nonzero(delays == 0),
          "; с задержкой:", np.count_nonzero(delays > 0))
    print(f"Средняя по всем доставкам: {delays.mean():.2f} мин; "
          f"максимальная: {delays.max():.2f} мин")
    print("Индекс максимальной задержки:", int(np.argmax(delays)))
    print("Индексы задержек > 20:", np.flatnonzero(long_delays))
    print(f"Средняя оценка всего датасета: {cleaned[:, 4].mean():.3f}")
    if long_delays.any():
        print(f"Средняя оценка при задержке > 20: {cleaned[long_delays, 4].mean():.3f}")
    else:
        print("Задержек > 20 нет; средняя оценка этой группы не определена.")

    condition_matrix = np.column_stack((
        cleaned[:, 1] > 90, cleaned[:, 3] > 15,
        cleaned[:, 4] < 3.5, delays > 20,
    ))
    problem_counts = condition_matrix.sum(axis=1)
    problem_indices = np.flatnonzero(problem_counts >= 2)
    print("\nПРОБЛЕМНЫЕ ДОСТАВКИ")
    print("Индексы:", problem_indices)
    print("Количество выполненных условий:", problem_counts[problem_indices])
    print("Полные строки:\n", cleaned[problem_indices])
    print_group_statistics("Проблемные", cleaned[problem_indices])
    print_group_statistics("Весь датасет", cleaned)

    correlations = np.corrcoef(numeric, rowvar=False)
    covariances = np.cov(numeric, rowvar=False, ddof=1)
    print("\nКОРРЕЛЯЦИОННАЯ МАТРИЦА")
    print_labeled_matrix(correlations, columns[:5])

    assert np.array_equal(source, source_before, equal_nan=True)
    assert not np.shares_memory(source, cleaned)
    print("\nПроверка: исходный массив не изменён.")
    return {
        "shape": source.shape,
        "audit": quality,
        "prepared_data": cleaned,
        "basic_statistics": statistics,
        "delivery_type_counts": type_counts,
        "groups": groups,
        "speed": speeds,
        "slowest_index": slowest,
        "fastest_index": fastest,
        "expected_time": expected,
        "delay": delays,
        "problem_count": problem_counts,
        "problem_indices": problem_indices,
        "correlation_matrix": correlations,
        "covariance_matrix": covariances,
    }


report = analyze_delivery_dataset(data)
assert np.array_equal(data, original_data, equal_nan=True)


АНАЛИТИЧЕСКИЙ ОТЧЁТ
Размер датасета: (120, 6)
Количество доставок: 120
Количество признаков: 6
Количество элементов: 720
dtype: float64 ; объём памяти: 5760 байт

АУДИТ ИСХОДНЫХ ДАННЫХ
NaN: 4
Координаты (индекс строки, индекс столбца):
[[ 2  1]
 [82  0]
 [85  4]
 [96  4]]
inf: 1
Координаты (индекс строки, индекс столбца):
[[101   2]]
-inf: 0
Координаты (индекс строки, индекс столбца):
[]
Конечные значения вне ограничений: 5
Координаты (индекс строки, индекс столбца):
[[  5   4]
 [ 25   2]
 [ 73   0]
 [108   1]
 [115   3]]
Подробный список проблем:
Доставка № 3 (индекс 2), признак duration, значение nan: пропуск
Доставка № 6 (индекс 5), признак rating, значение 8.0: нарушение ограничения
Доставка № 26 (индекс 25), признак order_amount, значение -3500.0: нарушение ограничения
Доставка № 74 (индекс 73), признак distance, значение -12.0: нарушение ограничения
Доставка № 83 (индекс 82), признак distance, значение nan: пропуск
Доставка № 86 (индекс 85), признак rating, значение nan: пропуск


## Итоговые аналитические выводы по варианту 5

1. **Данные нуждались в очистке:** из 120 доставок в 10 строках были
   ошибки — 4 пропуска, 1 бесконечность и 5 конечных недопустимых значений.
   После заполнения допустимыми средними сохранились все строки;
   исходный массив не изменён, пропуски и бесконечности отсутствуют.

2. **Экспресс-доставка в среднем быстрее на 16,12 минуты:** 41,17 против
   57,29 минуты. Однако её средний маршрут короче на 4,38 км, а среднее
   ожидание меньше на 0,77 минуты. Разницу нельзя целиком приписывать
   одному типу доставки. Средняя оценка экспресс выше лишь на 0,072 балла.

3. **Расстояние тесно связано со временем:** корреляция равна 0,8853.
   Это самая сильная положительная линейная связь среди пяти числовых
   признаков. Время доставки следует сравнивать с учётом длины маршрута.

4. **Длительное ожидание связано со снижением оценки:** корреляция
   ожидания и оценки равна −0,3400. Связь полного времени доставки
   с оценкой слабее (−0,1928). Это основание изучить ожидание курьера
   подробнее, но не доказательство причинного влияния.

5. **Скорость выполнения заказа в среднем составляет 18,51 км/ч**,
   медиана — 19,50 км/ч; значений свыше 60 км/ч нет. Минимум и максимум
   относятся к строкам с заполненными значениями, поэтому не подходят
   для надёжного ранжирования реальных курьеров.

6. **По учебной формуле 62 доставки имеют задержку, 58 выполнены без неё.**
   Средняя задержка по всем заказам — 4,40 минуты, максимальная —
   106,96 минуты (индекс 105). У двух заказов с задержкой свыше 20 минут
   оценка 4,540 против 4,495 в целом; такая малая и частично заполненная
   группа не позволяет утверждать, что задержки не снижают удовлетворённость.

7. **По правилу двух условий проблемной является одна доставка**
   (индекс 105): 29,02 км за 180 минут при ожидании всего 2,46 минуты
   и оценке 4,76. Её нужно проверить отдельно; она заметно повышает
   средние времена обычных и дорогих доставок. Высокая оценка не отменяет
   срабатывания формального правила, а правило не доказывает причину проблемы.

8. **Стоимость заказа почти не связана с расстоянием и временем:**
   корреляции равны 0,0076 и 0,0175. В верхних 10% стоимости среднее время
   больше общего (53,79 против 48,15 минуты), но квартильные средние
   не растут монотонно, а на верхнюю группу влияет доставка за 180 минут.
   Признаков принципиально иной организации доставки дорогих заказов
   по этому варианту не обнаружено.

9. **Средние чувствительны к необычным значениям и способу заполнения.**
   Медианная стоимость заказа 3 090,18 рубля ниже средней 3 597,73 рубля;
   на среднюю влияет заказ на 35 000 рублей. Необычные, но допустимые
   наблюдения сохранены, а выводы о скоростях, задержках и корреляциях
   учитывают ограничения заполнения пропусков средними.


---
# Что необходимо сдать

Один заполненный Jupyter Notebook `.ipynb`, содержащий:

1. номер варианта;
2. загрузку `delivery_data.npy`;
3. решение всех 10 заданий;
4. письменные ответы и выводы;
5. рабочую функцию `analyze_delivery_dataset(data)`;
6. не менее пяти итоговых аналитических выводов.

Все результаты должны воспроизводиться при повторной генерации того же варианта.


**Проверка выполнения:** указан вариант 5, решены все 10 заданий,
добавлены письменные ответы, функция итогового анализа и 9 выводов.
Для воспроизведения поместите ноутбук рядом с `delivery_data.npy`,
полученным генератором для варианта 5, и выполните все ячейки сверху вниз.
Для вычислений требуется только NumPy; pandas и другие библиотеки
анализа данных не используются.
